# Basic LangGraph Chatbot with Google Gemini

This notebook walks through a **minimal stateful chatbot** built with:

- **LangGraph** — graph of nodes that hold conversation state
- **LangChain** — message types + Gemini model wrapper
- **Google Gemini** — the LLM that generates replies
- **InMemorySaver** — remembers past turns for a given `thread_id`

**Flow each turn:** `START → chat_node (Gemini) → END`

Run the cells **from top to bottom**.

## 0. One-time setup

1. Copy `.env.example` → `.env` and put your Gemini key in it:
   ```bash
   cp .env.example .env
   # then edit .env and set GOOGLE_API_KEY=...
   ```
2. Get a free key: [Google AI Studio](https://aistudio.google.com/apikey)
3. Install deps (if you have not already):
   ```bash
   pip install -r requirements.txt
   ```

## 1. Imports & environment

Why each import exists is noted in comments in the next cell.

In [ ]:
# load_dotenv: reads key=value pairs from a local .env file into process env vars
# so we never hard-code the API key inside the notebook.
from dotenv import load_dotenv

# TypedDict: describes the shape of our graph state as a typed dictionary.
# Annotated: attaches metadata to a type — here we attach the add_messages reducer.
from typing import Annotated, TypedDict

# BaseMessage: parent type for HumanMessage / AIMessage / SystemMessage.
# HumanMessage: wraps text that came from the user.
from langchain_core.messages import BaseMessage, HumanMessage

# ChatGoogleGenerativeAI: LangChain wrapper around Google Gemini chat models.
# (Replaces ChatOpenAI from the original OpenAI version of this project.)
from langchain_google_genai import ChatGoogleGenerativeAI

# StateGraph / START / END: building blocks for a LangGraph workflow.
from langgraph.graph import StateGraph, START, END

# add_messages: reducer that APPENDS new messages to the list instead of replacing it.
from langgraph.graph.message import add_messages

# InMemorySaver: stores conversation state in RAM, keyed by thread_id.
from langgraph.checkpoint.memory import InMemorySaver

# Actually load .env now (must run before creating the LLM client).
load_dotenv()
print("Environment loaded. Make sure GOOGLE_API_KEY is set in your .env file.")

## 2. Create the Gemini model

`ChatGoogleGenerativeAI` reads `GOOGLE_API_KEY` from the environment automatically.

In [ ]:
# model="gemini-2.0-flash": fast, good default for chat demos.
# Alternatives: "gemini-1.5-flash", "gemini-1.5-pro", "gemini-2.5-flash", etc.
# If a model name 404s for your account, try another from AI Studio.
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

# Quick sanity check that the key + model work before we wire up LangGraph.
smoke = llm.invoke([HumanMessage(content="Reply with exactly: OK")])
print("Gemini smoke test:", smoke.content)

## 3. Define chat state

LangGraph needs a **state schema**. Ours only tracks `messages`.

The `Annotated[..., add_messages]` part is important: when a node returns
`{"messages": [new_msg]}`, LangGraph **appends** `new_msg` to the existing list
instead of overwriting everything.

In [ ]:
class ChatState(TypedDict):
    """
    Shared state that flows through every node in the graph.

    messages: full conversation so far (Human + AI turns).
    Annotated + add_messages => updates are merged/appended, not replaced.
    """
    messages: Annotated[list[BaseMessage], add_messages]

## 4. Define the chat node

A **node** is a function: `(state) → partial state update`.
This one sends the whole message list to Gemini and returns the reply.

In [ ]:
def chat_node(state: ChatState) -> dict:
    """
    One LLM call per graph run (i.e. per user turn).

    1. Read all messages already in state (prior turns + the new HumanMessage).
    2. Ask Gemini for the next assistant message.
    3. Return it inside a list so add_messages can append it to state.
    """
    messages = state["messages"]
    response = llm.invoke(messages)  # Gemini returns an AIMessage
    return {"messages": [response]}  # NOT a bare string — LangGraph expects messages

## 5. Build & compile the graph

```
START ──► chat_node ──► END
```

Compiling with a **checkpointer** enables memory: the same `thread_id`
keeps prior messages across `.invoke()` calls.

In [ ]:
# InMemorySaver keeps checkpoints in process memory.
# Restart the kernel → conversation history for this thread is gone.
checkpointer = InMemorySaver()

# StateGraph(ChatState) tells LangGraph what fields live in state.
graph = StateGraph(ChatState)

# Register our function as a named node in the graph.
graph.add_node("chat_node", chat_node)

# Wire the edges: always enter at chat_node, then finish this turn.
graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

# compile() produces a Runnable you can .invoke() / .stream().
# checkpointer=... enables multi-turn memory via config["configurable"]["thread_id"].
chatbot = graph.compile(checkpointer=checkpointer)

print("Graph compiled. Ready to chat.")

## 6. Single-turn example

Pass a `HumanMessage` and a `config` with a `thread_id`.
The checkpointer uses `thread_id` as the memory key.

In [ ]:
# Same thread_id = same conversation memory bucket.
CONFIG = {"configurable": {"thread_id": "notebook-thread-1"}}

# Only send the NEW user message. Prior turns are restored from the checkpointer.
response = chatbot.invoke(
    {"messages": [HumanMessage(content="Hi! Who are you in one short sentence?")]},
    config=CONFIG,
)

# response["messages"] is the FULL history for this thread after this turn.
# The last item is Gemini's latest reply.
ai_message = response["messages"][-1].content
print("Assistant:", ai_message)

## 7. Multi-turn memory check

Ask a follow-up that only makes sense if the bot remembers the previous turn.
We reuse the **same** `CONFIG` / `thread_id`.

In [ ]:
# Follow-up on the SAME thread — checkpointer injects prior messages before chat_node runs.
response2 = chatbot.invoke(
    {"messages": [HumanMessage(content="What was my previous message?")]},
    config=CONFIG,  # same thread_id as above
)

print("Assistant:", response2["messages"][-1].content)
print("\n--- Full message count in this thread:", len(response2["messages"]))

## 8. Interactive chat loop (run this cell)

Type messages in the notebook input prompt.
Type `quit`, `exit`, or `q` to stop.

Uses a **new** thread id so this interactive session starts fresh.

In [ ]:
# Fresh thread so the interactive loop does not mix with the demo cells above.
INTERACTIVE_CONFIG = {"configurable": {"thread_id": "notebook-interactive"}}

print("LangGraph + Gemini chatbot")
print("Type 'quit' / 'exit' / 'q' to stop.\n")

while True:
    # input() blocks until you type something in the notebook.
    user_input = input("You: ").strip()

    # Allow a few common exit words so you can leave the loop cleanly.
    if user_input.lower() in {"quit", "exit", "q"}:
        print("Bye!")
        break

    # Skip empty submits (user hit Enter with no text).
    if not user_input:
        continue

    # Run one graph turn; memory for INTERACTIVE_CONFIG is restored automatically.
    result = chatbot.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=INTERACTIVE_CONFIG,
    )

    # Print only the newest assistant content for a clean chat UX.
    print("Bot:", result["messages"][-1].content)
    print()

## 9. (Optional) Inspect full history for a thread

Useful when debugging what the checkpointer actually stored.

In [ ]:
# get_state returns the latest checkpoint for this thread_id.
snapshot = chatbot.get_state(INTERACTIVE_CONFIG)

print("Messages stored for thread 'notebook-interactive':\n")
for i, msg in enumerate(snapshot.values.get("messages", []), start=1):
    # msg.type is usually "human" or "ai"
    role = msg.type
    # Truncate long replies so the notebook stays readable.
    content = msg.content if isinstance(msg.content, str) else str(msg.content)
    preview = content if len(content) <= 200 else content[:200] + "..."
    print(f"{i}. [{role}] {preview}")

## 10. Optional Streamlit UI

The same backend lives in `langgraph_backend.py`. To open a browser chat UI:

```bash
streamlit run streamlit_app.py
```

You do **not** need Streamlit to use this notebook.

## What you learned

| Piece | Role |
| --- | --- |
| `ChatState` | Schema for conversation state |
| `add_messages` | Appends new messages instead of overwriting |
| `chat_node` | Calls Gemini once per user turn |
| `InMemorySaver` | Remembers state per `thread_id` |
| `config.thread_id` | Which conversation memory bucket to use |
| `ChatGoogleGenerativeAI` | Gemini instead of OpenAI |

Next ideas: add a system prompt node, tool-calling nodes, or swap `InMemorySaver` for a SQLite/Postgres checkpointer.